In [ ]:
import importlib
import pandas as pd
import algoritmos_clasificacion as t
from math import log2

# Recargar el módulo para que incluya las nuevas funciones
importlib.reload(t)

S = t.cargarDataset("data_dia1")

# Usando len()
num_filas = len(S)
print("Cantidad de filas:", num_filas)





Cantidad de filas: 20300
Cantidad de filas: 20300


In [44]:
#PRUEBAS

# Recargar el módulo para que incluya las nuevas funciones
importlib.reload(t)

#S OK
#t.entropia(S, "dia_lluvioso") OK
# -21524/rows * log2(21524/rows) -  (rows-21524)/rows * log2((rows-21524)/rows) #sin nuls

#print(S[S["tmax_carrasco"] == "muy_calido"]) OK
#print(S.isna().sum()) #OK
#print(t.ganancia(S, "tmax_carrasco", "dia_lluvioso")) OK
S = S[["hum_carrasco", "hum_salto","tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto", "dia_lluvioso"]]
A = ["tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto"]


print("Gain (Entropía) ",t.get_gains(S,A, "dia_lluvioso"))  
print("Info Ratio (Gini) ",t.get_max_irs(S,A, "dia_lluvioso"))
print("Gain Ratio con SplitInfo ",t.get_gain_ratios(S,A, "dia_lluvioso")) #no va favorecer tanto a los atributos con muchos valores

#a = t.obtener_max_gan(S, A, "dia_lluvioso") 
#S[a].value_counts().index.tolist()


arbol = t.generacion_ID3(S, A, "dia_lluvioso", 0.003, "gain_ratio") #min_info_gain=0.2
dot = t.graficar_arbol(arbol)
dot.render("ARBOL gain ratio", format="png", view=True)

# arbol = t.generacion_ID3(S, A, "dia_lluvioso", 0.003, "gini") #min_info_gain=0.2
# dot = t.graficar_arbol(arbol)
# dot.render("ARBOL gini", format="png", view=True)



Gain (Entropía)  {'tmax_carrasco': np.float64(0.004606710114631651), 'tmin_carrasco': np.float64(0.005301112440531219), 'tmax_salto': np.float64(0.006037070928483379), 'tmin_salto': np.float64(0.008405816764690988), 'amplitud_carrasco': np.float64(0.0005477097946905429), 'amplitud_salto': np.float64(0.0019902290040552106)}
Info Ratio (Gini)  {'tmax_carrasco': np.float64(0.0021644373336521805), 'tmin_carrasco': np.float64(0.002463394032351296), 'tmax_salto': np.float64(0.00279506245844513), 'tmin_salto': np.float64(0.0039273670042618836), 'amplitud_carrasco': np.float64(0.00026280294063113496), 'amplitud_salto': np.float64(0.0009740464677159277)}
Gain Ratio con SplitInfo  {'tmax_carrasco': np.float64(0.0021593177288315958), 'tmin_carrasco': np.float64(0.002508408462122846), 'tmax_salto': np.float64(0.0028304650210733404), 'tmin_salto': np.float64(0.003956723076183831), 'amplitud_carrasco': np.float64(0.00028225629979395045), 'amplitud_salto': np.float64(0.0010258863106834522)}


'ARBOL gain ratio.png'

In [45]:
gains = t.get_gains(S, A, "dia_lluvioso")  
irs   = t.get_max_irs(S, A, "dia_lluvioso")    
gr_winner = t.get_gain_ratios(S, A, "dia_lluvioso")  

# Unimos en un DataFrame
df_comp = pd.DataFrame({
    "Gain (Entropía)": pd.Series(gains),
    "Impurity Reduction (Gini)": pd.Series(irs),
    "Gain Ratio (Gain y Split info)": pd.Series(gr_winner)
})

# Get a random row from S
random_row = S.sample(1).iloc[0]  
predecir = t.predecir(arbol, random_row)
predecir


(eog:15819): EOG-CRITICAL **: 15:11:00.420: eog_image_get_file: assertion 'EOG_IS_IMAGE (img)' failed

(eog:15819): GLib-GIO-CRITICAL **: 15:11:00.420: g_file_equal: assertion 'G_IS_FILE (file1)' failed


'no'

In [46]:
fila_test = pd.Series({
    "tmin_salto": "muy_frio",
    "tmax_salto": "calido",
})

prediccion = t.predecir(arbol, fila_test)
print(prediccion)


si


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import algoritmos_clasificacion as t
import pandas as pd
from sklearn.model_selection import KFold
import algoritmos_clasificacion as t
import importlib
importlib.reload(t)
def crear_lags_concat(X, y, nd):
    """
    Genera variables lag (días previos) hasta 'nd' para cada columna de X.
    Devuelve X_lag e y_lag con las filas sincronizadas y sin NaN.
    """
    lags = {}
    for col in X.columns:
        for d in range(1, nd+1):
            lags[f"{col}_lag{d}"] = X[col].shift(d)

    X_lag = pd.concat([X, pd.DataFrame(lags)], axis=1)

    # Eliminar filas con NaN generadas por shift
    mask = ~X_lag.isna().any(axis=1)
    return X_lag[mask], y[mask]


# Hiperparámetros a probar con Bayes
m_values = [0.5, 1, 2, 5]
nd_values = [3] # [1, 2, 3]

resultados_bayes = {}

S = t.cargarDataset("data_dia1")

X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
       "amplitud_carrasco","amplitud_salto",
       "dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
       "dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom",
       "vis_carrasco_prom","vis_salto_prom"]]

y = S["dia_lluvioso"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

for m in m_values:
    for nd in nd_values:
        print(f"\n=== Evaluando Naive Bayes con m={m}, nd={nd} ===")
        
        X_train_lag, y_train_lag = crear_lags_concat(X_train, y_train, nd)
        X_test_lag, y_test_lag = crear_lags_concat(X_test, y_test, nd)

        # Crear modelo
        model = t.NaiveBayesClassifier(m=m)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train_lag, y_train_lag, cv=kf)
        resultados_bayes[(m, nd)] = cv_scores
        
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train_lag, y_train_lag)
        y_pred = model.predict(X_test_lag)

        # Accuracy
        acc_test = accuracy_score(y_test_lag, y_pred)
        # F1-score
        f1_test = f1_score(y_test_lag, y_pred, average='binary', pos_label='si')
        # Recall
        recall_test = recall_score(y_test_lag, y_pred, average='binary', pos_label='si')
        # Matriz de confusión
        cm = confusion_matrix(y_test_lag, y_pred, labels=['si','no'])
        
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)
# Resumen final de Naive Bayes para los distintos m y nd
print("\nRESUMEN NAIVE BAYES (Accuracy promedio CV ± desviación)")
for (m, nd), scores in resultados_bayes.items():
    print(f"m={m}, nd={nd} : {scores.mean():.4f} ± {scores.std():.4f}")